# Pre-Trained Large Language Models

**AI3 · NLP Deep Dive** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~15 min

## Introduction

In all of our previous tasks, we were training a neural network to perform a certain task using labeled dataset. With large transformer models, such as BERT, we use language modelling in self-supervised fashion to build a language model, which is then specialized for specific downstream task with further domain-specific training. However, it has been demonstrated that large language models can also solve many tasks without ANY domain-specific training. A family of models capable of doing that is called **GPT**: Generative Pre-Trained Transformer.



## Text Generation and Perplexity

The idea of a neural network being able to do general tasks without downstream training is presented in [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) paper. The main idea is the many other tasks can be modeled using **text generation**, because understanding text essentially means being able to produce it. Because the model is trained on a huge amount of text that encompasses human knowledge, it also becomes knowledgeable about wide variety of subjects.

> Understanding and being able to produce text also entails knowing something about the world around us. People  also learn by reading to the large extent, and GPT network is similar in this respect.

Text generation networks work by predicting probability of the next word $$P(w_N)$$ However, unconditional probability of the next word equals to the frequency of the this word in the text corpus. GPT is able to give us **conditional probability** of the next word, given the previous ones: $$P(w_N | w_{n-1}, ..., w_0)$$

> You can read more about probabilities in our [Data Science for Beginers Curriculum](https://github.com/microsoft/Data-Science-For-Beginners/tree/main/1-Introduction/04-stats-and-probability)

Quality of language generating model can be defined using **perplexity**. It is intrinsic metric that allows us to measure the model quality without any task-specific dataset. It is based on the notion of *probability of a sentence* - the model assigns high probability to a sentence that is likely to be real (i.e. the model is not **perplexed** by it), and low probability to sentences that make less sense (eg. *Can it does what?*). When we give our model sentences from real text corpus, we would expect them to have high probability, and low **perplexity**. Mathematically, it is defined as normalized inverse probability of the test set:
$$
\mathrm{Perplexity}(W) = \sqrt[N]{1\over P(W_1,...,W_N)}
$$ 

**You can experiment with text generation using GPT-powered text editor from Hugging Face (service closed)**. In this editor, you start writing your text, and pressing **[TAB]** will offer you several completion options. If they are too short, or you are not satisfied with them - press [TAB] again, and you will have more options, including longer pieces of text.



## GPT is a Family

GPT is not a single model, but rather a collection of models developed and trained by [OpenAI](https://openai.com). 

Under the GPT models, we have:

| [GPT-2](https://huggingface.co/docs/transformers/model_doc/gpt2#openai-gpt2) | [GPT 3](https://openai.com/research/language-models-are-few-shot-learners) | [GPT-4](https://openai.com/gpt-4) |
| -- | -- | -- |
|Language model with upto 1.5 billion parameters. | Language model with up to 175 billion parameters | 100T parameters and accepts both image and text inputs and outputs text. |


The GPT-3 and GPT-4 models are available [as a cognitive service from Microsoft Azure](https://azure.microsoft.com/en-us/services/cognitive-services/openai-service/#overview?WT.mc_id=academic-77998-cacaste), and as [OpenAI API](https://openai.com/api/).



## Prompt Engineering

Because GPT has been trained on a vast volumes of data to understand language and code, they provide outputs in response to inputs (prompts). Prompts are GPT inputs or queries whereby one provides instructions to models on tasks they next completed. To elicit a desired outcome, you need the most effective prompt which involves selecting the right words, formats, phrases or even symbols. This approach is [Prompt Engineering](https://learn.microsoft.com/en-us/shows/ai-show/the-basics-of-prompt-engineering-with-azure-openai-service?WT.mc_id=academic-77998-bethanycheum)

[This documentation](https://learn.microsoft.com/en-us/semantic-kernel/prompt-engineering/?WT.mc_id=academic-77998-bethanycheum) provides you with more information on prompt engineering.



## Example Notebook: Playing with OpenAI-GPT

Continue your learning in the following notebooks:

* [Generating text with OpenAI-GPT and Hugging Face Transformers](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/20-LangModels/GPT-PyTorch.ipynb)



## Conclusion

New general pre-trained language models do not only model language structure, but also contain vast amount of natural language. Thus, they can be effectively used to solve some NLP tasks in zero-shop or few-shot settings.



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

1. [Perplexity from log-probs](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m04-perplexity)
2. [Token frequency counter](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m04-token-counts)
3. [Instruction-following checker](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m04-format-check)

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Perplexity from log-probs

●○○ · ~6 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m04-perplexity)

> Perplexity is the standard language-model metric: the exponential of the average **negative** log-likelihood, PPL = exp(−mean(log p(token))). A perfect model assigns log p = 0 → PPL = 1; worse models get higher values.

Implement `perplexity(logprobs)`: given the natural log-probabilities the model assigned to the true tokens, return exp(−mean(logprobs)).

<details><summary>Hint</summary>

One line: math.exp(-np.mean(logprobs)). Mind the minus sign.

</details>


In [ ]:
import numpy as np

def perplexity(logprobs):
    # your code
    return float(np.exp(np.mean(logprobs)))  # wrong — sign

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(3, 30))\n    return {\"logprobs\": -rng.uniform(0.05, 4.0, n)}\n\ndef perplexity(logprobs):\n    return float(math.exp(-float(np.mean(logprobs))))", "perplexity", 20)

### Task 2 — Token frequency counter

●○○ · ~8 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m04-token-counts)

> The simplest tokenizer splits text on whitespace. Counting token frequencies is the basis of Bag-of-Words, TF-IDF and vocab building. Case matters — 'The' and 'the' are the same word, so lowercase first.

Implement `token_counts(text)` → dict mapping each **lowercased** whitespace-separated token to its count.

<details><summary>Hint</summary>

Loop over text.lower().split() and use counts.get(w, 0) + 1 — or collections.Counter.

</details>


In [ ]:
import numpy as np

def token_counts(text):
    # your code
    return {}

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    vocab = [\"the\", \"a\", \"cat\", \"dog\", \"sits\", \"runs\", \"llm\", \"ai\", \"is\", \"on\"]\n    n = int(rng.integers(4, 20))\n    words = [str(w) for w in rng.choice(vocab, n)]\n    words = [w.upper() if rng.random() < 0.3 else w for w in words]\n    return {\"text\": \" \".join(words)}\n\ndef token_counts(text):\n    counts = {}\n    for w in text.lower().split():\n        counts[w] = counts.get(w, 0) + 1\n    return counts", "token_counts", 20)

### Task 3 — Instruction-following checker

●●○ · ~12 min · [Open in the trainer →](https://abzlitochka.github.io/ai-olympiad-curriculum/#ex=m04-format-check)

> Benchmarks for instruction following (IFEval-style) grade outputs with *verifiable* constraints: must start with a prefix, must contain required phrases, must not contain forbidden ones, must respect a word limit. Each check is one boolean — combine them with AND.

Implement `follows_spec(text, prefix, required, forbidden, max_words)` → True if and only if **all** hold:
- `text` starts with `prefix`
- every string in `required` is a substring of `text`
- no string in `forbidden` is a substring of `text`
- `len(text.split())` ≤ `max_words`

<details><summary>Hint</summary>

Four conditions ANDed. `s in text` tests substring; str.startswith for the prefix; text.split() for words.

</details>


In [ ]:
import numpy as np

def follows_spec(text, prefix, required, forbidden, max_words):
    # your code
    return True

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    pool = [\"model\", \"answer\", \"value\", \"result\", \"is\", \"the\", \"42\", \"ok\"]\n    prefix = str(rng.choice([\"ANSWER:\", \"RESULT:\", \"DONE:\"]))\n    n = int(rng.integers(3, 9))\n    text = \" \".join(str(w) for w in rng.choice(pool, n))\n    if rng.random() < 0.6:\n        text = prefix + \" \" + text\n    required = [str(w) for w in rng.choice(pool, int(rng.integers(1, 3)))]\n    forbidden = [str(w) for w in rng.choice([\"sorry\", \"cannot\", \"maybe\"], int(rng.integers(1, 3)))]\n    if rng.random() < 0.3:\n        text = text + \" \" + str(rng.choice(forbidden))\n    # substring-vs-whole-word traps: required phrases with spaces (present in\n    # text) and forbidden substrings hidden inside longer words — a\n    # word-in-text.split() implementation must fail both ways\n    if rng.random() < 0.35:\n        phrase = str(rng.choice([\"is the\", \"the 42\", \"answer is\"]))\n        required.append(phrase)\n        if rng.random() < 0.7:\n            text = text + \" \" + phrase\n    if rng.random() < 0.35:\n        text = text + \" \" + str(rng.choice([\"history\", \"analysis\", \"isotope\"]))\n        forbidden.append(str(rng.choice([\"is\", \"anal\", \"isot\"])))\n    return {\"text\": text, \"prefix\": prefix, \"required\": required,\n            \"forbidden\": forbidden, \"max_words\": int(rng.integers(2, 10))}\n\ndef follows_spec(text, prefix, required, forbidden, max_words):\n    if not text.startswith(prefix):\n        return False\n    if any(r not in text for r in required):\n        return False\n    if any(f in text for f in forbidden):\n        return False\n    return len(text.split()) <= max_words", "follows_spec", 20)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/20-LangModels)